# Introducción a la Frontera Eficiente de Markowitz

**Universidad Internacional de La Rioja (UNIR)**  
**Maestría en Ciencias Computacionales y Matemáticas Aplicadas**  
**Profesor:** Dr. Rodolfo Rafael Medina Ramírez

---

## 🎯 Objetivos

1. Comprender los conceptos fundamentales de la teoría moderna de carteras
2. Calcular rendimientos y riesgos de carteras
3. Implementar optimización básica
4. Visualizar la frontera eficiente

In [ ]:
# Importar bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
pd.options.display.float_format = '{:.4f}'.format

print('✅ Bibliotecas importadas')

## Datos de Ejemplo

Trabajaremos con 5 activos.

In [ ]:
# Definir activos
activos = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']
rendimientos_esperados = np.array([0.15, 0.12, 0.13, 0.18, 0.20])
volatilidades = np.array([0.25, 0.22, 0.20, 0.30, 0.35])

matriz_correlacion = np.array([
    [1.00, 0.75, 0.80, 0.65, 0.55],
    [0.75, 1.00, 0.78, 0.70, 0.50],
    [0.80, 0.78, 1.00, 0.68, 0.52],
    [0.65, 0.70, 0.68, 1.00, 0.60],
    [0.55, 0.50, 0.52, 0.60, 1.00]
])

matriz_covarianzas = np.outer(volatilidades, volatilidades) * matriz_correlacion
tasa_libre_riesgo = 0.03

print('Datos configurados')

## Funciones de Optimización

In [ ]:
def rendimiento_cartera(pesos, rendimientos):
    return np.dot(pesos, rendimientos)

def volatilidad_cartera(pesos, cov_matrix):
    return np.sqrt(np.dot(pesos.T, np.dot(cov_matrix, pesos)))

def sharpe_ratio(pesos, rendimientos, cov_matrix, rf):
    ret = rendimiento_cartera(pesos, rendimientos)
    vol = volatilidad_cartera(pesos, cov_matrix)
    return (ret - rf) / vol

print('✅ Funciones definidas')

In [ ]:
def cartera_tangente(rendimientos, cov_matrix, rf):
    n = len(rendimientos)
    
    def objetivo(w):
        return -sharpe_ratio(w, rendimientos, cov_matrix, rf)
    
    restricciones = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    limites = tuple((0, 1) for _ in range(n))
    pesos_iniciales = np.array([1/n] * n)
    
    resultado = minimize(objetivo, pesos_iniciales, method='SLSQP',
                        bounds=limites, constraints=restricciones)
    
    pesos = resultado.x
    
    return {
        'pesos': pesos,
        'rendimiento': rendimiento_cartera(pesos, rendimientos),
        'volatilidad': volatilidad_cartera(pesos, cov_matrix),
        'sharpe': sharpe_ratio(pesos, rendimientos, cov_matrix, rf)
    }

# Calcular
tangente = cartera_tangente(rendimientos_esperados, matriz_covarianzas, tasa_libre_riesgo)

print('⭐ CARTERA TANGENTE')
print(f"Rendimiento: {tangente['rendimiento']*100:.2f}%")
print(f"Volatilidad: {tangente['volatilidad']*100:.2f}%")
print(f"Sharpe: {tangente['sharpe']:.4f}")
print('\nComposición:')
for i, activo in enumerate(activos):
    if tangente['pesos'][i] > 0.01:
        print(f"  {activo}: {tangente['pesos'][i]*100:.2f}%")

## Visualización

Generamos carteras aleatorias para visualizar la frontera.

In [ ]:
# Generar carteras aleatorias
np.random.seed(42)
n_carteras = 5000
n_activos = len(activos)

rendimientos_carteras = []
volatilidades_carteras = []
sharpes_carteras = []

for _ in range(n_carteras):
    pesos = np.random.random(n_activos)
    pesos /= np.sum(pesos)
    
    ret = rendimiento_cartera(pesos, rendimientos_esperados)
    vol = volatilidad_cartera(pesos, matriz_covarianzas)
    sr = sharpe_ratio(pesos, rendimientos_esperados, matriz_covarianzas, tasa_libre_riesgo)
    
    rendimientos_carteras.append(ret)
    volatilidades_carteras.append(vol)
    sharpes_carteras.append(sr)

rendimientos_carteras = np.array(rendimientos_carteras)
volatilidades_carteras = np.array(volatilidades_carteras)
sharpes_carteras = np.array(sharpes_carteras)

print(f'✅ Generadas {n_carteras} carteras')

In [ ]:
# Gráfico
plt.figure(figsize=(14, 8))

scatter = plt.scatter(volatilidades_carteras * 100, rendimientos_carteras * 100,
                     c=sharpes_carteras, cmap='viridis', alpha=0.3, s=10)
plt.colorbar(scatter, label='Ratio de Sharpe')

plt.scatter(volatilidades * 100, rendimientos_esperados * 100,
           marker='o', s=150, c='red', edgecolors='black', linewidth=2,
           label='Activos', zorder=3)

plt.scatter(tangente['volatilidad'] * 100, tangente['rendimiento'] * 100,
           marker='*', s=500, c='gold', edgecolors='black', linewidth=2,
           label='Tangente', zorder=5)

plt.xlabel('Volatilidad (%)', fontsize=12, fontweight='bold')
plt.ylabel('Rendimiento (%)', fontsize=12, fontweight='bold')
plt.title('Frontera Eficiente', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

**Universidad Internacional de La Rioja (UNIR)**  
*Última actualización: Agosto 2025*